# 5. K-Nearest Neighbors (KNN) Classification

Thuật toán KNN phân loại điểm dữ liệu bằng cách tính khoảng cách (distance) tới `K` điểm gần nhất. Kết quả phụ thuộc vào biểu quyết đa số (majority voting).

## Tầm quan trọng của Scaling
Vì dựa trên khoảng cách (thường là khoảng cách Euclidean), nếu một đặc trưng có thang đo lớn hơn, nó sẽ lấn át các đặc trưng khác. Preprocessing pipeline đã xử lý vấn đề này bằng `StandardScaler`.

## Hyperparameters
- `n_neighbors (K)`: Số lượng hàng xóm.
- `weights`: `uniform` (mọi điểm có sức nặng như nhau) hoặc `distance` (điểm gần hơn có trọng số lớn hơn).

In [ ]:
import sys
sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from src.preprocessing import get_preprocessing_pipeline
from src.models.knn_classifier import get_knn_classifier
from src.metrics import calculate_metrics

df = pd.read_csv("../data/heart_cleveland_upload.csv")
X = df.drop("condition", axis=1)
y = df["condition"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
pipeline = Pipeline([
    ("preprocessor", get_preprocessing_pipeline()),
    ("model", get_knn_classifier())
])

param_grid = {
    "model__n_neighbors": [1, 3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

## Đánh giá K vs CV Score

In [ ]:
import pandas as pd
results = pd.DataFrame(grid.cv_results_)
plt.figure(figsize=(8, 5))
for weight in ["uniform", "distance"]:
    subset = results[results["param_model__weights"] == weight]
    plt.plot(subset["param_model__n_neighbors"], subset["mean_test_score"], marker="o", label=f"weights={weight}")
plt.title("KNN: K vs Validation Accuracy")
plt.xlabel("n_neighbors (K)")
plt.ylabel("CV Accuracy")
plt.legend()
plt.show()

In [ ]:
y_pred = grid.predict(X_test)
y_score = grid.predict_proba(X_test)[:, 1]
print("Test Metrics:", calculate_metrics(y_test, y_pred, y_score))